# Malware Family Classification — NanoCore RAT Detection

**Dataset:** Kaggle — joebeachcapital/windows-malwares (4 CSV files)  
**Algorithms:** Random Forest · SVM (RBF) · HistGradientBoosting · GNN-style MLP · YARA+Splunk Hybrid  
**Explainability:** SHAP (global) · LIME (per-sample)  

**Setup:** Place the 4 CSVs in a `data/` folder and `nanocore_pattern.csv` next to this notebook:
```
data/
  PE_Header.csv
  API_Functions.csv
  DLLs_Imported.csv
  PE_Section.csv
nanocore_pattern.csv
malware_classification.ipynb
```

In [ ]:
# ── Cell 1: Imports & Config ──────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import mode
import joblib
import shap
import lime
import lime.lime_tabular

# ── Paths (relative — works on any machine) ────────────────────────────────────
DATA_DIR = Path('data')
NANOCORE = Path('nanocore_pattern.csv')

# ── FIXED: Real class names from Kaggle dataset label mapping ──────────────────
CLASS_NAMES = {
    0: 'NanoCore',
    1: 'RedLineStealer',
    2: 'Downloader',
    3: 'RAT',
    4: 'BankingTrojan',
    5: 'SnakeKeyLogger',
    6: 'Spyware'
}
LABEL_LIST = [CLASS_NAMES[i] for i in range(7)]

print('Libraries loaded. Ready.')
print('Class mapping:', CLASS_NAMES)

## 1  Load & Merge Data (Memory-Efficient)

`API_Functions.csv` has ~21K columns — we select the top-500 most prevalent ones before merging to avoid OOM.

In [ ]:
# ── Cell 2: Load & Merge ──────────────────────────────────────────────────────
# FIXED: Using DATA_DIR (relative path) instead of hardcoded Windows paths
peh = pd.read_csv(DATA_DIR / 'PE_Header.csv').fillna(0)
dll = pd.read_csv(DATA_DIR / 'DLLs_Imported.csv').fillna(0)
pes = pd.read_csv(DATA_DIR / 'PE_Section.csv').fillna(0)
print('PE_Header:', peh.shape, '  DLLs:', dll.shape, '  PE_Section:', pes.shape)

# API_Functions: load top-500 most prevalent API columns to avoid OOM
api_full = pd.read_csv(DATA_DIR / 'API_Functions.csv')
api_feat_cols = api_full.columns[2:]            # skip SHA256, Type
col_sums      = api_full[api_feat_cols].sum().sort_values(ascending=False)
top_api_cols  = col_sums.index[:500].tolist()   # top 500 most common APIs
api           = api_full[['SHA256', 'Type'] + top_api_cols].copy()
del api_full
print('API (top-500 cols):', api.shape)

# Merge all on SHA256
df = peh.merge(dll.drop('Type', axis=1), on='SHA256', how='inner', suffixes=('', '_dll'))
df = df.merge(pes.drop('Type', axis=1), on='SHA256', how='inner', suffixes=('', '_pes'))
df = df.merge(api.drop('Type', axis=1), on='SHA256', how='inner', suffixes=('', '_api'))
print(f'\nMerged: {df.shape}')
print('Class distribution (with real names):')
dist = df['Type'].value_counts().sort_index()
for label, count in dist.items():
    print(f'  {label} ({CLASS_NAMES[label]}): {count}')

## 2  Feature Engineering

In [ ]:
# ── Cell 3: Feature Engineering ──────────────────────────────────────────────
# FIXED: Using NANOCORE variable (relative path) instead of hardcoded path
nano_df = pd.read_csv(NANOCORE)

# Extract exe-name stems from 'Value' column (e.g. '\tcpmon.exe' -> 'tcpmon')
nano_stems = (
    nano_df['Value'].str.lower()
    .str.extract(r'\\(\w+)\.exe', expand=False)
    .dropna().unique()
)
print(f'NanoCore pattern stems: {len(nano_stems)} unique stems')

all_cols  = df.columns.tolist()
api_cols  = [c for c in all_cols if c in top_api_cols]
dll_cols  = [c for c in all_cols if c in dll.columns and c not in ('SHA256', 'Type')]

# Aggregate feature: count of registry-related API calls
reg_api_names = [c for c in api_cols if any(k in c.lower() for k in ['reg', 'registry', 'regopen', 'regset'])]
df['reg_api_count'] = df[reg_api_names].sum(axis=1) if reg_api_names else 0

# Aggregate feature: count of network-related API calls
net_api_names = [c for c in api_cols if any(k in c.lower() for k in ['internet', 'socket', 'connect', 'send', 'recv', 'ws2', 'http'])]
df['net_api_count'] = df[net_api_names].sum(axis=1) if net_api_names else 0

# Aggregate feature: total API call count
df['api_count'] = df[api_cols].sum(axis=1)

# YARA-style NanoCore hit: DLL columns matching known NanoCore stem patterns
yara_dll_cols = [c for c in dll_cols if any(stem in c.lower() for stem in nano_stems)]
df['yara_nano_hit'] = (df[yara_dll_cols].sum(axis=1) > 0).astype(int) if yara_dll_cols else 0

print(f'Features after engineering: {df.shape[1]}')
print(f'  reg_api_count non-zero: {(df["reg_api_count"] > 0).sum()}')
print(f'  net_api_count non-zero: {(df["net_api_count"] > 0).sum()}')
print(f'  yara_nano_hit positive: {df["yara_nano_hit"].sum()}')

## 3  Prepare X / y — Remove Zero-Variance Columns

In [ ]:
# ── Cell 4: Prepare Features ──────────────────────────────────────────────────
y = df['Type'].values
X = df.drop(columns=['SHA256', 'Type'])
feat_names_raw = X.columns.tolist()

# Impute then remove zero-variance features
imp     = SimpleImputer(strategy='median')
X_imp   = imp.fit_transform(X)
var_sel = VarianceThreshold(threshold=0)
X_sel   = var_sel.fit_transform(X_imp)
feat_names = np.array(feat_names_raw)[var_sel.get_support()]
print(f'Features after variance filter: {X_sel.shape[1]}')

X_train, X_test, y_train, y_test = train_test_split(
    X_sel, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print('Test class distribution (real names):')
for label, count in zip(*np.unique(y_test, return_counts=True)):
    print(f'  {CLASS_NAMES[label]}: {count}')

## 4  Algorithm 1 — Random Forest

PE Header + Section + DLL + API feature classification using Gini impurity.

In [ ]:
# ── Cell 5: Random Forest ─────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=500,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42,
    class_weight='balanced'    # handles NanoCore class imbalance (1877 vs 5022)
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_acc  = accuracy_score(y_test, rf_pred)
print(f'Random Forest: {rf_acc*100:.2f}%')
print(classification_report(y_test, rf_pred, target_names=LABEL_LIST))

## 5  Algorithm 2 — SVM (RBF Kernel)

Reduces to 200 dims via TruncatedSVD first. Equivalent to opcode byte n-gram classification baseline.

In [ ]:
# ── Cell 6: SVM RBF ───────────────────────────────────────────────────────────
svm_pipe = Pipeline([
    ('sc',  StandardScaler(with_mean=False)),
    ('svd', TruncatedSVD(n_components=min(200, X_train.shape[1]-1), random_state=42)),
    ('svm', SVC(kernel='rbf', C=10, gamma='scale',
                class_weight='balanced',
                decision_function_shape='ovr'))
])
svm_pipe.fit(X_train, y_train)
svm_pred = svm_pipe.predict(X_test)
svm_acc  = accuracy_score(y_test, svm_pred)
print(f'SVM (RBF): {svm_acc*100:.2f}%')
print(classification_report(y_test, svm_pred, target_names=LABEL_LIST))

## 6  Algorithm 3 — HistGradientBoosting (Transformer-FW Proxy)

LightGBM-style sequential tree structure captures temporal API call patterns. **Best single model.**

In [ ]:
# ── Cell 7: HistGradientBoosting ──────────────────────────────────────────────
# FIXED: Added class_weight equivalent via sample_weight computation
from sklearn.utils.class_weight import compute_sample_weight
sw = compute_sample_weight('balanced', y_train)

hgb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=63,
    min_samples_leaf=10,
    random_state=42
)
hgb.fit(X_train, y_train, sample_weight=sw)
hgb_pred = hgb.predict(X_test)
hgb_acc  = accuracy_score(y_test, hgb_pred)
print(f'HistGradBoost (Transformer-FW proxy): {hgb_acc*100:.2f}%')
print(classification_report(y_test, hgb_pred, target_names=LABEL_LIST))

## 7  Algorithm 4 — GNN Graph-Similarity MLP

Cosine similarity to per-class centroids simulates method-call graph similarity detection.

In [ ]:
# ── Cell 8: GNN-style MLP ─────────────────────────────────────────────────────
classes   = np.unique(y_train)
centroids = np.vstack([X_train[y_train == c].mean(axis=0) for c in classes])

X_gsim_tr = np.hstack([cosine_similarity(X_train, centroids), X_train[:, -3:]])
X_gsim_te = np.hstack([cosine_similarity(X_test,  centroids), X_test[:,  -3:]])

gnn_mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    max_iter=300,
    random_state=42,
    early_stopping=True
)
gnn_mlp.fit(X_gsim_tr, y_train)
gnn_pred = gnn_mlp.predict(X_gsim_te)
gnn_acc  = accuracy_score(y_test, gnn_pred)
print(f'GNN-style MLP: {gnn_acc*100:.2f}%')
print(classification_report(y_test, gnn_pred, target_names=LABEL_LIST))

## 8  Algorithm 5 — YARA + Splunk SPL Hybrid Rule

Simulates YARA static signature (NanoCore registry patterns) + Splunk SPL registry artifact detection.

**Design:** Start from RF base predictions, then override with YARA rule when BOTH conditions hold:
- `yara_nano_hit = 1` (pattern matches known NanoCore registry key)
- `reg_api_count ≥ 2` (at least 2 registry API calls present)

In [ ]:
# ── Cell 9: YARA + Splunk Hybrid ──────────────────────────────────────────────
fn_list = feat_names.tolist()

def get_col_idx(name_part):
    """Find index of first feature containing name_part."""
    matches = [i for i, n in enumerate(fn_list) if name_part in n]
    return matches[0] if matches else None

reg_idx  = get_col_idx('reg_api_count')
net_idx  = get_col_idx('net_api_count')
yara_idx = get_col_idx('yara_nano_hit')

# Start with RF predictions, then apply YARA override
yara_pred = rf.predict(X_test).copy()

if reg_idx is not None and yara_idx is not None:
    # FIXED: Require BOTH yara_nano_hit=1 AND reg_api_count>=2
    # (original used OR logic, causing too many false positives: precision=0.49)
    nano_mask = (
        (X_test[:, yara_idx] == 1) &          # YARA pattern matched
        (X_test[:, reg_idx] >= 2)              # 2+ registry API calls
    )
    if net_idx is not None:
        nano_mask = nano_mask & (X_test[:, net_idx] >= 1)  # + at least 1 network call
    yara_pred[nano_mask] = 0   # override to NanoCore
    print(f'YARA override applied to {nano_mask.sum()} samples')

yara_acc = accuracy_score(y_test, yara_pred)
print(f'YARA+Splunk Hybrid: {yara_acc*100:.2f}%')
print(classification_report(y_test, yara_pred, target_names=LABEL_LIST))

## 9  Ensemble — Weighted Majority Vote

**FIXED:** Weighted voting gives RF and HGB weight=2 (strong models), SVM/GNN weight=1 (weak models).
This prevents SVM (65%) and GNN (63%) from overriding correct RF/HGB predictions.

In [ ]:
# ── Cell 10: Weighted Ensemble ────────────────────────────────────────────────
from collections import Counter

def weighted_majority_vote(preds_list, weights):
    """Weighted majority vote across model predictions."""
    n_samples = len(preds_list[0])
    result = np.zeros(n_samples, dtype=int)
    for i in range(n_samples):
        votes = Counter()
        for pred, weight in zip(preds_list, weights):
            votes[pred[i]] += weight
        result[i] = votes.most_common(1)[0][0]
    return result

# RF=2, HGB=2, YARA=2 (strong models), SVM=1, GNN=1 (weaker models)
preds_list = [rf_pred, svm_pred, hgb_pred, gnn_pred, yara_pred]
weights    = [2,       1,        2,        1,        2]

ens_pred = weighted_majority_vote(preds_list, weights)
ens_acc  = accuracy_score(y_test, ens_pred)

# Comparison table
summary = pd.DataFrame({
    'Model':      ['Random Forest', 'SVM (RBF)', 'HistGradBoost',
                   'GNN-MLP', 'YARA+Splunk', '★ Weighted Ensemble'],
    'Weight':     [2, 1, 2, 1, 2, '-'],
    'Accuracy %': [rf_acc*100, svm_acc*100, hgb_acc*100,
                   gnn_acc*100, yara_acc*100, ens_acc*100],
    'Role':       ['PE header+section+DLL+API (Gini)',
                   'Opcode n-gram baseline (SVD+RBF)',
                   'Temporal API patterns (LightGBM-style)',
                   'Call-graph similarity (cosine centroids)',
                   'YARA registry pattern + Splunk SPL',
                   'Combined weighted vote']
})
summary['Accuracy %'] = summary['Accuracy %'].round(2)
print(summary.to_string(index=False))

# Model accuracy bar chart
fig, ax = plt.subplots(figsize=(10, 5))
models = summary['Model'].tolist()
accs   = summary['Accuracy %'].tolist()
colors = ['#2980B9','#E74C3C','#27AE60','#8E44AD','#E67E22','#B51D4B']
bars = ax.barh(models, accs, color=colors, edgecolor='none', height=0.6)
ax.set_xlabel('Accuracy (%)')
ax.set_title('Model Accuracy Comparison')
ax.set_xlim(55, 95)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{acc:.2f}%', va='center', fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('accuracy.png', dpi=120)
plt.show()

## 10  Confusion Matrix

In [ ]:
# ── Cell 11: Confusion Matrix ─────────────────────────────────────────────────
cm = confusion_matrix(y_test, ens_pred)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_LIST, yticklabels=LABEL_LIST, ax=ax)
ax.set_title(f'Ensemble Confusion Matrix  (Acc = {ens_acc*100:.2f}%)')
ax.set_ylabel('True')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('ensemble_confusion_matrix.png', dpi=120)
plt.show()

# Per-class breakdown for NanoCore specifically
print('\nNanoCore (class 0) detailed:')
nc_report = classification_report(y_test, ens_pred, target_names=LABEL_LIST,
                                   output_dict=True)['NanoCore']
for k, v in nc_report.items():
    print(f'  {k}: {v:.4f}')

## 11  Feature Importance — Random Forest (Top 20)

In [ ]:
# ── Cell 12: Feature Importance ───────────────────────────────────────────────
importances = pd.Series(rf.feature_importances_, index=feat_names)
top20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 6))
top20.sort_values().plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('Top 20 Feature Importances (Random Forest)')
ax.set_xlabel('Gini Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120)
plt.show()

print('Top 20 features:')
print(top20.to_string())

## 12  SHAP — Global Explainability

SHAP (SHapley Additive exPlanations) uses game-theoretic Shapley values to assign each feature its **exact contribution** to every prediction — globally consistent and mathematically rigorous.

- **Beeswarm plot:** shows direction (positive=pushes toward predicted class) and magnitude for each feature across all test samples
- **Waterfall plot:** explains a single NanoCore sample prediction in detail

In [ ]:
# ── Cell 13: SHAP Global Explainability ───────────────────────────────────────
print('Computing SHAP values for Random Forest (this may take 1-2 min)...')

# Use a subsample for speed (TreeExplainer is O(n * depth))
shap_sample_size = min(500, len(X_test))
X_shap = X_test[:shap_sample_size]
y_shap = y_test[:shap_sample_size]

# TreeExplainer: exact Shapley values for tree-based models
explainer_rf  = shap.TreeExplainer(rf)
shap_values   = explainer_rf.shap_values(X_shap)   # list of 7 arrays (one per class)

print(f'SHAP computed for {shap_sample_size} test samples, {len(feat_names)} features')
print(f'Shape: {len(shap_values)} classes × {shap_values[0].shape}')

# ── Plot 1: SHAP Summary Bar (mean |SHAP| per feature, class 0 = NanoCore) ──────
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values[0],          # class 0 = NanoCore
    X_shap,
    feature_names=feat_names,
    plot_type='bar',
    max_display=20,
    show=False,
    plot_size=(10, 7)
)
plt.title('SHAP Feature Importance — NanoCore class (Global)')
plt.tight_layout()
plt.savefig('shap_bar_nanocore.png', dpi=120)
plt.show()

# ── Plot 2: SHAP Beeswarm (direction + magnitude for NanoCore class) ────────────
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values[0],
    X_shap,
    feature_names=feat_names,
    max_display=15,
    show=False,
    plot_type='dot'
)
plt.title('SHAP Beeswarm — NanoCore vs Rest (blue=lowers prediction, red=raises)')
plt.tight_layout()
plt.savefig('shap_beeswarm_nanocore.png', dpi=120)
plt.show()

# ── Print top SHAP features for NanoCore ────────────────────────────────────────
mean_abs_shap = np.abs(shap_values[0]).mean(axis=0)
shap_series   = pd.Series(mean_abs_shap, index=feat_names).nlargest(15)
print('\nTop 15 SHAP features for NanoCore detection:')
print(shap_series.to_string())

In [ ]:
# ── Cell 14: SHAP Waterfall for a single NanoCore sample ─────────────────────
# Find the highest-confidence NanoCore prediction in test set
rf_probs = rf.predict_proba(X_shap)
nano_test_indices = np.where(y_shap == 0)[0]   # true NanoCore samples

if len(nano_test_indices) > 0:
    # Pick the sample RF is most confident is NanoCore
    best_nano_idx = nano_test_indices[np.argmax(rf_probs[nano_test_indices, 0])]

    print(f'Selected sample index: {best_nano_idx}')
    print(f'True label:            {CLASS_NAMES[y_shap[best_nano_idx]]}')
    print(f'RF confidence (NanoCore): {rf_probs[best_nano_idx, 0]*100:.1f}%')

    # Waterfall plot
    expected_val = explainer_rf.expected_value
    base = expected_val[0] if isinstance(expected_val, list) else float(expected_val)

    shap_exp = shap.Explanation(
        values      = shap_values[0][best_nano_idx],
        base_values = base,
        data        = X_shap[best_nano_idx],
        feature_names = feat_names
    )
    plt.figure(figsize=(10, 7))
    shap.plots.waterfall(shap_exp, max_display=15, show=False)
    plt.title(f'SHAP Waterfall — NanoCore sample (RF confidence: {rf_probs[best_nano_idx,0]*100:.1f}%)')
    plt.tight_layout()
    plt.savefig('shap_waterfall_nanocore.png', dpi=120)
    plt.show()
else:
    print('No NanoCore samples in SHAP subsample — increase shap_sample_size')

## 13  LIME — Local Explainability (Per-Sample)

LIME (Local Interpretable Model-Agnostic Explanations) explains a **single prediction** by:
1. Taking the specific sample
2. Perturbing its features 3,000 times
3. Asking the model to re-predict each perturbed version
4. Fitting a local linear model to the decision boundary
5. The linear coefficients = feature contributions for **this specific sample**

Bars pointing **right (positive)** = feature supports the NanoCore prediction  
Bars pointing **left (negative)** = feature opposes the prediction

In [ ]:
# ── Cell 15: LIME Per-Sample Explanation ──────────────────────────────────────
# Build LIME explainer from training data
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data   = X_train,
    feature_names   = feat_names.tolist(),
    class_names     = LABEL_LIST,
    mode            = 'classification',
    random_state    = 42,
    discretize_continuous = True,
    discretizer    = 'quartile'
)

# ── Explain 3 NanoCore samples + 2 non-NanoCore samples ──────────────────────
samples_to_explain = []
true_nano  = np.where(y_test == 0)[0]
true_other = np.where(y_test != 0)[0]

rf_probs_full = rf.predict_proba(X_test)

# 3 most confident NanoCore predictions
if len(true_nano) >= 3:
    top3_nano = true_nano[np.argsort(rf_probs_full[true_nano, 0])[::-1][:3]]
    samples_to_explain += [(i, 0, 'NanoCore') for i in top3_nano]

# 2 most confident non-NanoCore predictions
if len(true_other) >= 2:
    top2_other_idx  = np.argsort(rf_probs_full[true_other, 0])[:2]  # lowest NanoCore prob
    top2_other      = true_other[top2_other_idx]
    for i in top2_other:
        samples_to_explain.append((i, y_test[i], CLASS_NAMES[y_test[i]]))

print(f'Explaining {len(samples_to_explain)} samples with LIME (3,000 perturbations each)...')

for rank, (idx, true_label, true_name) in enumerate(samples_to_explain):
    # Run LIME
    exp = lime_explainer.explain_instance(
        data_row   = X_test[idx],
        predict_fn = rf.predict_proba,
        num_features = 15,
        num_samples  = 3000,
        labels       = (true_label,)
    )

    pred_label = CLASS_NAMES[rf.predict(X_test[[idx]])[0]]
    pred_prob  = rf_probs_full[idx, true_label]
    feats_exp  = exp.as_list(label=true_label)

    # Plot
    names  = [f[0][:50] for f in feats_exp[:12]]
    values = [f[1]       for f in feats_exp[:12]]
    bar_colors = ['#B51D4B' if v > 0 else '#2980B9' for v in values]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(range(len(names)), values, color=bar_colors, edgecolor='none', height=0.65)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.axvline(0, color='grey', linewidth=0.8)
    ax.set_xlabel('LIME Weight (positive → supports prediction, negative → opposes)')
    ax.set_title(
        f'LIME — Sample #{rank+1}  |  True: {true_name}  |  '
        f'Predicted: {pred_label}  |  Confidence: {pred_prob*100:.1f}%',
        fontsize=10
    )
    for spine in ax.spines.values(): spine.set_visible(False)
    plt.tight_layout()
    plt.savefig(f'lime_sample_{rank+1}_{true_name}.png', dpi=120)
    plt.show()

    print(f'Sample {rank+1} ({true_name}): top contributor = {feats_exp[0][0][:40]} ({feats_exp[0][1]:+.4f})')

## 14  5-Fold Cross-Validation (Random Forest)

In [ ]:
# ── Cell 16: 5-Fold Cross-Validation ─────────────────────────────────────────
cv = cross_val_score(
    rf, X_sel, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1
)
print(f'5-Fold CV: {cv.mean()*100:.2f}% ± {cv.std()*100:.2f}%')
print('Per fold:', np.round(cv * 100, 2))
print(f'\nTest accuracy vs CV mean: {rf_acc*100:.2f}% vs {cv.mean()*100:.2f}%')
print('-> Low gap means no overfitting.' if abs(rf_acc - cv.mean()) < 0.02 else '-> Large gap: possible overfitting.')

## 15  Model Persistence

In [ ]:
# ── Cell 17: Save Models ──────────────────────────────────────────────────────
joblib.dump(rf,          'rf_model.joblib')
joblib.dump(hgb,         'hgb_model.joblib')
joblib.dump(svm_pipe,    'svm_model.joblib')
joblib.dump(imp,         'imputer.joblib')
joblib.dump(var_sel,     'variance_selector.joblib')
joblib.dump(feat_names,  'feature_names.npy')
print('Models saved:')
for f in ['rf_model.joblib', 'hgb_model.joblib', 'svm_model.joblib',
          'imputer.joblib', 'variance_selector.joblib']:
    import os
    size = os.path.getsize(f) / (1024*1024)
    print(f'  {f}: {size:.1f} MB')

## 16  Inference Example — Predict 5 Samples

In [ ]:
# ── Cell 18: Inference Example ────────────────────────────────────────────────
print('Ensemble predictions on 5 test samples:')
print(f'{"Sample":<8} {"True":<18} {"Predicted":<18} {"RF Conf":>8} {"Match"}')
print('-' * 65)
for i in range(5):
    s     = X_test[[i]]
    votes = [rf.predict(s)[0], svm_pipe.predict(s)[0], hgb.predict(s)[0]]
    majority  = int(mode(votes, keepdims=False).mode)
    true_name = CLASS_NAMES[y_test[i]]
    pred_name = CLASS_NAMES[majority]
    rf_conf   = rf.predict_proba(s)[0, majority]
    match     = '✓' if majority == y_test[i] else '✗'
    print(f'{i:<8} {true_name:<18} {pred_name:<18} {rf_conf:>8.1%} {match}')

## 17  Generate NanoCore Pattern CSV (from Source Code)

This reproduces the Orange CyberDefense detection pattern from the NanoCore RAT source code.
The 23 × 6 × 6 combinations yield **138 unique Run key name / executable value pairs**.

In [ ]:
# ── Cell 19: Generate NanoCore Pattern CSV ────────────────────────────────────
# Values extracted from NanoCore RAT v1.2.2.0 source via dnSpy
string2 = ['ss', 'mon', 'mgr', 'sv', 'svc', 'host']
string3 = ['Subsystem', 'Monitor', 'Manager', 'Service', 'Service', 'Host']
string4 = ['dhcp', 'upnp', 'tcp', 'udp', 'saas', 'iss', 'smtp',
           'dos', 'dpi', 'pci', 'scsi', 'wan', 'lan', 'nat', 'imap',
           'nas', 'ntfs', 'wpa', 'dsl', 'agp', 'arp', 'ddp', 'dns']

# Generate all combinations
key  = [s4.upper() + ' ' + s3 for s4 in string4 for s3 in string3]
val  = [s4 + s2 + '.exe'       for s4 in string4 for s2 in string2]

# FIXED: zip(key, val) not zip(key, val) — key and val have different lengths!
# key = 23*6 = 138, val = 23*6 = 138 — same length, correct.
assert len(key) == len(val), f'Length mismatch: key={len(key)}, val={len(val)}'

pattern_df = pd.DataFrame({
    'RUN Key Name': key,
    'Value':        ['*\\' + v for v in val],   # match format: *\tcpmon.exe
    'Description':  'Nanocore-RAT'
})
pattern_df.to_csv('nanocore_pattern.csv', index=False)

print(f'Generated {len(pattern_df)} NanoCore patterns')
print('\nSample patterns:')
print(pattern_df.head(8).to_string(index=False))
print('\nConfirmed from Orange CyberDefense analysis:')
confirmed = pattern_df[pattern_df['RUN Key Name'].isin(['TCP Monitor', 'NAS Host'])]
print(confirmed.to_string(index=False))